In [1]:
#cell1
import argparse
import time
import os  # 新增：用于拼接文件路径
from typing import List, Tuple

import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import sys

sys.path.append("/kaggle/input/models/zn1350/chess-game/pytorch/default/7")

from features import FEATURE_DIM, MOVE_FEAT_DIM, _move_to_feat
from nn_evaluator import MicroDualNet
from chess_game import ChessGame

# 固定随机种子（保证可重复性）
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 训练参数
DATA_PATH = "/kaggle/input/models/zn1350/chess-game/pytorch/default/7"   # 包含所有 .npy 文件的文件夹路径
EPOCHS = 100
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
VAL_RATIO = 0.1
OUT_PATH = "/kaggle/working/micro_dual.pth"       # 输出模型路径
NUM_WORKERS = 2

In [2]:
#cell2
# 训练目标分数的缩放系数：搜索原始分数量级在几百~几千（子力分值），
# 缩放到与 tanh 价值头相近的量级，避免策略头 loss 数值过大主导训练。
_SCORE_SCALE = 1000.0

In [3]:
#cell3
class SelfPlayDataset(Dataset):
    """
    每个样本：(feature_vec, move_feats(该局面全部走法), target_scores(归一化后)， z)
    走法特征在 __getitem__ 时按需计算（用 board 还原一个临时 ChessGame，不做磁盘 IO，开销很小）。
    从 data_dir 下的独立 .npy 文件加载。
    """

    def __init__(self, data_dir: str):
        self.features = np.load(os.path.join(data_dir, "features.npy")).astype(np.float32)
        self.boards = np.load(os.path.join(data_dir, "boards.npy")).astype(np.int8)
        self.move_scores = np.load(os.path.join(data_dir, "move_scores.npy"), allow_pickle=True)
        self.moves_order = np.load(os.path.join(data_dir, "moves_order.npy"), allow_pickle=True)
        self.z_values = np.load(os.path.join(data_dir, "z_values.npy")).astype(np.float32)
        assert len(self.features) == len(self.z_values)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        feat = self.features[idx]
        board = self.boards[idx]
        moves = self.moves_order[idx]
        scores = self.move_scores[idx]
        z = self.z_values[idx]

        g = ChessGame.__new__(ChessGame)
        g.board = [list(row) for row in board]

        move_feats = np.stack([_move_to_feat(g, tuple(m)) for m in moves], axis=0).astype(np.float32)
        target_scores = (np.asarray(scores, dtype=np.float32)) / _SCORE_SCALE

        return (
            torch.from_numpy(feat),
            torch.from_numpy(move_feats),
            torch.from_numpy(target_scores),
            torch.tensor(z, dtype=torch.float32),
        )

In [4]:
#cell4
def collate_fn(batch):
    """变长走法数需要 padding 到该 batch 内的最大长度，并生成 mask 以在 loss 中忽略 padding。"""
    feats, move_feats_list, target_scores_list, zs = zip(*batch)

    feats = torch.stack(feats, dim=0)                     # (B, 60)
    zs = torch.stack(zs, dim=0).unsqueeze(-1)              # (B, 1)

    max_moves = max(mf.shape[0] for mf in move_feats_list)
    batch_size = len(move_feats_list)

    move_feats = torch.zeros((batch_size, max_moves, MOVE_FEAT_DIM), dtype=torch.float32)
    target_scores = torch.zeros((batch_size, max_moves), dtype=torch.float32)
    mask = torch.zeros((batch_size, max_moves), dtype=torch.float32)

    for i, (mf, ts) in enumerate(zip(move_feats_list, target_scores_list)):
        n = mf.shape[0]
        move_feats[i, :n] = mf
        target_scores[i, :n] = ts
        mask[i, :n] = 1.0

    return feats, move_feats, target_scores, mask, zs


def _split_train_val(data_dir: str, val_ratio: float = 0.1):
    dataset = SelfPlayDataset(data_dir)
    n = len(dataset)
    n_val = max(1, int(n * val_ratio)) if n > 1 else 0
    n_train = n - n_val
    if n_val == 0:
        return dataset, dataset
    return torch.utils.data.random_split(dataset, [n_train, n_val])


def _masked_mse(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    diff2 = (pred - target) ** 2 * mask
    denom = mask.sum().clamp_min(1.0)
    return diff2.sum() / denom



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}, GPU 数量: {torch.cuda.device_count() if torch.cuda.is_available() else 0}")

print(f"加载数据: {DATA_PATH}")
train_set, val_set = _split_train_val(DATA_PATH, VAL_RATIO)
print(f"训练集: {len(train_set)} 个局面, 验证集: {len(val_set)} 个局面")

train_loader = DataLoader(
    train_set, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=NUM_WORKERS,
)
val_loader = DataLoader(
    val_set, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=NUM_WORKERS,
)

model = MicroDualNet(feature_dim=FEATURE_DIM, move_feat_dim=MOVE_FEAT_DIM).to(device)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print(f"使用 DataParallel，GPU 数: {torch.cuda.device_count()}")

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
use_amp = device.type == "cuda"
scaler = GradScaler(enabled=use_amp)

best_val_loss = float("inf")
t_start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0
    train_batches = 0

    for feats, move_feats, target_scores, mask, zs in train_loader:
        feats = feats.to(device, non_blocking=True)
        move_feats = move_feats.to(device, non_blocking=True)
        target_scores = target_scores.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)
        zs = zs.to(device, non_blocking=True)

        optimizer.zero_grad()
        with autocast(device_type=device.type, enabled=use_amp):
            v_pred, scores_pred = model(feats, move_feats)
            loss_v = nn.functional.mse_loss(v_pred, zs)
            loss_p = _masked_mse(scores_pred, target_scores, mask)
            loss = loss_v + loss_p

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss_sum += float(loss.item())
        train_batches += 1

    model.eval()
    val_loss_sum = 0.0
    val_batches = 0
    with torch.no_grad():
        for feats, move_feats, target_scores, mask, zs in val_loader:
            feats = feats.to(device, non_blocking=True)
            move_feats = move_feats.to(device, non_blocking=True)
            target_scores = target_scores.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)
            zs = zs.to(device, non_blocking=True)

            v_pred, scores_pred = model(feats, move_feats)
            loss_v = nn.functional.mse_loss(v_pred, zs)
            loss_p = _masked_mse(scores_pred, target_scores, mask)
            loss = loss_v + loss_p

            val_loss_sum += float(loss.item())
            val_batches += 1

    train_loss = train_loss_sum / max(1, train_batches)
    val_loss = val_loss_sum / max(1, val_batches)
    elapsed = time.time() - t_start
    remaining = elapsed / epoch * (EPOCHS - epoch)
    print(
        f"[epoch {epoch}/{EPOCHS}] train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
        f"耗时={elapsed/60:.1f}分钟 预估剩余={remaining/60:.1f}分钟"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        state_dict = (model.module if isinstance(model, nn.DataParallel) else model).state_dict()
        torch.save({"state_dict": state_dict}, OUT_PATH)
        print(f"  -> 验证损失改善，已保存模型到 {OUT_PATH}")

total_time = time.time() - t_start
print(f"训练完成，最佳验证损失 {best_val_loss:.4f}，总耗时 {total_time/60:.1f} 分钟")

使用设备: cuda, GPU 数量: 2
加载数据: /kaggle/input/models/zn1350/chess-game/pytorch/default/7
训练集: 162000 个局面, 验证集: 18000 个局面
使用 DataParallel，GPU 数: 2


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/linear.py:134: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:330.)
  return F.linear(input, self.weight, self.bias)


[epoch 1/100] train_loss=111.5830 val_loss=104.0088 耗时=0.5分钟 预估剩余=46.6分钟
  -> 验证损失改善，已保存模型到 /kaggle/working/micro_dual.pth
[epoch 2/100] train_loss=92.1358 val_loss=85.5269 耗时=0.9分钟 预估剩余=44.5分钟
  -> 验证损失改善，已保存模型到 /kaggle/working/micro_dual.pth
[epoch 3/100] train_loss=77.8570 val_loss=78.0414 耗时=1.3分钟 预估剩余=43.4分钟
  -> 验证损失改善，已保存模型到 /kaggle/working/micro_dual.pth
[epoch 4/100] train_loss=74.1681 val_loss=76.2673 耗时=1.8分钟 预估剩余=42.6分钟
  -> 验证损失改善，已保存模型到 /kaggle/working/micro_dual.pth
[epoch 5/100] train_loss=73.2795 val_loss=75.4875 耗时=2.2分钟 预估剩余=42.1分钟
  -> 验证损失改善，已保存模型到 /kaggle/working/micro_dual.pth
[epoch 6/100] train_loss=72.7797 val_loss=75.3792 耗时=2.6分钟 预估剩余=41.5分钟
  -> 验证损失改善，已保存模型到 /kaggle/working/micro_dual.pth
[epoch 7/100] train_loss=72.2928 val_loss=74.9802 耗时=3.1分钟 预估剩余=41.1分钟
  -> 验证损失改善，已保存模型到 /kaggle/working/micro_dual.pth
[epoch 8/100] train_loss=71.4284 val_loss=73.3032 耗时=3.5分钟 预估剩余=40.5分钟
  -> 验证损失改善，已保存模型到 /kaggle/working/micro_dual.pth
[epoch 9/100] train_loss=70.50